# Data Exploration for ML Trading Strategy

This notebook explores the historical market data for the S&P 500 and sector ETFs.

## Table of Contents
1. Load and Inspect Data
2. Summary Statistics
3. Price Trends Visualization
4. Returns Analysis
5. Volatility Analysis
6. Correlation Analysis
7. Feature Distribution

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import config
from src.data_sourcing import DataDownloader

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load and Inspect Data

In [ ]:
# Initialize data downloader
downloader = DataDownloader(config.START_DATE, config.END_DATE, config.DATA_DIR)

# Load data
market_data = downloader.load_data('market_data.parquet')
sector_data = downloader.load_data('sectors_raw.parquet')

print(f"Market data shape: {market_data.shape}")
print(f"Sector data shape: {sector_data.shape}")
print(f"\nDate range: {market_data.index.min()} to {market_data.index.max()}")
print(f"Total trading days: {len(market_data)}")

In [ ]:
# Display first few rows
print("Market Data (S&P 500 + VIX):")
display(market_data.head())

print("\nSector ETF Data:")
display(sector_data.head())

## 2. Summary Statistics

In [ ]:
# Market data summary
print("Market Data Summary Statistics:")
display(market_data.describe())

# Check for missing values
print("\nMissing Values:")
print(market_data.isnull().sum())

In [ ]:
# Sector data summary
print("Sector ETF Summary Statistics:")
display(sector_data.describe())

## 3. Price Trends Visualization

In [ ]:
# S&P 500 price chart
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: S&P 500 Close Price
axes[0].plot(market_data.index, market_data['close'], linewidth=1.5, color='#2E86AB')
axes[0].set_title('S&P 500 Index - Historical Prices', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot 2: VIX
axes[1].plot(market_data.index, market_data['vix'], linewidth=1.5, color='#E63946')
axes[1].set_title('VIX (Volatility Index)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('VIX Level', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sector ETF prices (normalized to 100)
normalized_sectors = (sector_data / sector_data.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 8))

for column in normalized_sectors.columns:
    ax.plot(normalized_sectors.index, normalized_sectors[column], label=column, linewidth=1.5)

ax.set_title('Sector ETF Performance (Normalized to 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Normalized Price', fontsize=12)
ax.legend(loc='best', ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Returns Analysis

In [ ]:
# Calculate returns
sp500_returns = market_data['close'].pct_change()
sector_returns = sector_data.pct_change()

# Summary statistics for returns
print("S&P 500 Returns Statistics:")
print(f"Mean daily return: {sp500_returns.mean():.4%}")
print(f"Std dev (daily): {sp500_returns.std():.4%}")
print(f"Annualized return: {sp500_returns.mean() * 252:.2%}")
print(f"Annualized volatility: {sp500_returns.std() * np.sqrt(252):.2%}")
print(f"Sharpe ratio (rf=2%): {(sp500_returns.mean() * 252 - 0.02) / (sp500_returns.std() * np.sqrt(252)):.4f}")

In [ ]:
# Returns distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(sp500_returns.dropna(), bins=50, alpha=0.7, color='#2E86AB', edgecolor='black')
axes[0].axvline(sp500_returns.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0].set_title('S&P 500 Daily Returns Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Return', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Q-Q plot
from scipy import stats
stats.probplot(sp500_returns.dropna(), dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (Normal Distribution)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sector returns comparison
sector_annual_returns = sector_returns.mean() * 252
sector_annual_vol = sector_returns.std() * np.sqrt(252)

# Create DataFrame
sector_stats = pd.DataFrame({
    'Annual Return': sector_annual_returns,
    'Annual Volatility': sector_annual_vol,
    'Sharpe Ratio': (sector_annual_returns - 0.02) / sector_annual_vol
})

print("Sector Performance Metrics:")
display(sector_stats.sort_values('Sharpe Ratio', ascending=False))

## 5. Volatility Analysis

In [ ]:
# Rolling volatility
rolling_vol_20 = sp500_returns.rolling(window=20).std() * np.sqrt(252)
rolling_vol_60 = sp500_returns.rolling(window=60).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(rolling_vol_20.index, rolling_vol_20.values, label='20-day Rolling Vol', linewidth=1.5)
ax.plot(rolling_vol_60.index, rolling_vol_60.values, label='60-day Rolling Vol', linewidth=1.5)
ax.set_title('S&P 500 Rolling Volatility (Annualized)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Volatility', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = sector_returns.corr()

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)

ax.set_title('Sector ETF Correlation Matrix (Returns)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# VIX vs S&P 500 correlation
vix_sp500_corr = market_data[['close', 'vix']].pct_change().corr()
print("VIX vs S&P 500 Correlation:")
print(vix_sp500_corr)

## 7. Feature Distribution

Let's explore the engineered features if they exist.

In [ ]:
# Try to load features if available
try:
    features_df = downloader.load_data('features.parquet')
    
    if features_df is not None:
        print(f"Features shape: {features_df.shape}")
        print(f"\nFeature columns:")
        print(list(features_df.columns))
        
        # Display feature statistics
        display(features_df.describe())
        
        # Plot some key features
        key_features = ['rsi_14', 'macd', 'bb_percent_b', 'garch_volatility']
        existing_features = [f for f in key_features if f in features_df.columns]
        
        if existing_features:
            fig, axes = plt.subplots(len(existing_features), 1, figsize=(14, 4*len(existing_features)))
            
            if len(existing_features) == 1:
                axes = [axes]
            
            for i, feature in enumerate(existing_features):
                axes[i].plot(features_df.index, features_df[feature], linewidth=1)
                axes[i].set_title(f'{feature}', fontsize=12, fontweight='bold')
                axes[i].set_ylabel('Value', fontsize=11)
                axes[i].grid(True, alpha=0.3)
            
            axes[-1].set_xlabel('Date', fontsize=11)
            plt.tight_layout()
            plt.show()
        
        # Target distribution
        if 'target' in features_df.columns:
            target_dist = features_df['target'].value_counts(normalize=True)
            print("\nTarget Distribution:")
            print(f"Down (0): {target_dist[0]:.2%}")
            print(f"Up (1): {target_dist[1]:.2%}")
    else:
        print("Features not yet created. Run feature engineering first.")
        
except Exception as e:
    print(f"Could not load features: {str(e)}")
    print("Run the feature engineering step first.")

## Summary

This notebook explored the historical market data:
- S&P 500 shows general upward trend with periodic drawdowns
- VIX spikes during market stress periods
- Sector ETFs show varying performance and correlations
- Returns are approximately normally distributed with fat tails
- Volatility clustering is evident

Next steps:
1. Feature engineering (technical indicators, GARCH)
2. Model training (LSTM)
3. Backtesting and evaluation